# Notebook 03 — Avaliação: Benchmarks e Análise de SLOs

## Objetivos

1. Avaliar métricas de SLO (P50, P95, P99) conforme seção 'Auto-scaling e SLOs'
2. Analisar compliance com SLOs de produção para diferentes configurações
3. Comparar impacto de batch size e limites de tokens no throughput
4. Avaliar trade-off entre throughput e latência (Tabela 1 do material)
5. Analisar cenários de capacity planning com ZeRO e paralelismo híbrido

## Contexto Teórico

> *"Service Level Objectives (SLOs) definem contratos de performance que sistemas
> de serving devem cumprir. Para inferência de LLMs, métricas típicas incluem
> P99 latência (tipicamente < 500ms), throughput mínimo (req/s), e
> disponibilidade (99.9%+)."* — Seção 'Estratégias de Auto-scaling e SLOs'

**Referências:**
- Kwon et al. (2023). Efficient Memory Management for LLM Serving. SOSP.
- Yu et al. (2022). Orca. OSDI.
- Rajbhandari et al. (2020). ZeRO. SC 2020.
- Narayanan et al. (2021). Efficient Large-Scale Language Model Training. SC 2021.

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocessing import WorkloadPreprocessor
from src.model import (
    ContinuousBatchingScheduler,
    ScheduledRequest,
    amdahl_speedup,
    gustafson_speedup,
)
from src.training import InferenceRunner, ServingConfig
from src.evaluation import ServingMetricsEvaluator
from src.utils import load_metrics, save_metrics, compute_memory_savings

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

evaluator = ServingMetricsEvaluator(output_dir='../outputs/figures')

print('Imports carregados.')

## 1. Carregamento de Dados e Métricas Anteriores

In [ ]:
# Carrega workload e métricas do notebook anterior
df = WorkloadPreprocessor.load_data('../data/raw/dataset.csv')

# Carrega métricas salvas (se disponíveis)
try:
    batching_metrics = load_metrics('../outputs/logs/batching_comparison.json')
    print('Métricas de batching carregadas do notebook 02.')
    print(f'  Continuous batching — avg latency: '
          f'{batching_metrics["metrics"]["continuous_batching"]["avg_latency_steps"]:.1f} steps')
    print(f'  Static batching    — avg latency: '
          f'{batching_metrics["metrics"]["static_batching"]["avg_latency_steps"]:.1f} steps')
except FileNotFoundError:
    print('Métricas não encontradas. Execute o notebook 02 primeiro.')
    batching_metrics = None

print(f'\nWorkload: {len(df)} requisições')

## 2. Análise de SLOs para Diferentes Configurações

Conforme seção *Estratégias de Auto-scaling e SLOs*:

> *"Scaling baseado em queue depth, além de utilização de GPU, captura melhor
> cenários onde latência degrada antes de CPUs saturarem."*

Vamos avaliar como diferentes tamanhos de batch afetam a conformidade com SLOs.

In [ ]:
# Sweep de configurações de batch size
batch_sizes = [4, 8, 16, 32, 64]
slo_results = []
df_sample = df.head(300)

for bs in batch_sizes:
    config = ServingConfig(max_batch_size=bs, max_total_tokens=bs * 256)
    runner = InferenceRunner(config)
    result = runner.run_continuous_batching(df_sample, max_steps=10000)
    
    # Coleta latências individuais
    latencies = []
    for req in runner.scheduler.completed:
        latencies.append(req.finish_step - req.arrival_step)
    latencies = np.array(latencies)
    
    slo_check = evaluator.check_slo_compliance(
        latencies, slo_p99_ms=500.0, slo_throughput_min=0.1,
        total_time_s=result['total_steps'],
    )
    
    slo_results.append({
        'batch_size': bs,
        'total_steps': result['total_steps'],
        'avg_latency': result['avg_latency_steps'],
        'p50_latency': result['p50_latency_steps'],
        'p95_latency': result['p95_latency_steps'],
        'p99_latency': result['p99_latency_steps'],
        'throughput': result['throughput_req_per_step'],
        'batch_utilization': result.get('avg_batch_utilization', 0),
    })

slo_df = pd.DataFrame(slo_results)
print('Métricas por Batch Size:')
slo_df.round(2)

In [ ]:
# Visualização: trade-off latência vs. throughput
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# P99 Latency vs Batch Size
ax = axes[0]
ax.plot(slo_df['batch_size'], slo_df['p99_latency'], 'o-',
        color='red', linewidth=2, markersize=8, label='P99')
ax.plot(slo_df['batch_size'], slo_df['p95_latency'], 's-',
        color='orange', linewidth=2, markersize=8, label='P95')
ax.plot(slo_df['batch_size'], slo_df['p50_latency'], '^-',
        color='green', linewidth=2, markersize=8, label='P50')
ax.set_xlabel('Max Batch Size')
ax.set_ylabel('Latência (steps)')
ax.set_title('Percentis de Latência vs. Batch Size')
ax.legend()
ax.grid(True, alpha=0.3)

# Throughput vs Batch Size
ax = axes[1]
ax.plot(slo_df['batch_size'], slo_df['throughput'], 'o-',
        color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Max Batch Size')
ax.set_ylabel('Throughput (req/step)')
ax.set_title('Throughput vs. Batch Size')
ax.grid(True, alpha=0.3)

# Total steps vs Batch Size
ax = axes[2]
ax.bar(slo_df['batch_size'].astype(str), slo_df['total_steps'],
       color='steelblue', alpha=0.8)
ax.set_xlabel('Max Batch Size')
ax.set_ylabel('Total Steps para Completar')
ax.set_title('Eficiência Total')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Trade-off Latência vs. Throughput — Configurações de Serving',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/03_slo_tradeoffs.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Distribuição de Latência com SLO

Análise detalhada da distribuição de latência para a configuração principal,
verificando conformidade com SLOs típicos de produção.

In [ ]:
# Executa configuração padrão e analisa distribuição de latência
config_main = ServingConfig(max_batch_size=16, max_total_tokens=4096)
runner_main = InferenceRunner(config_main)
result_main = runner_main.run_continuous_batching(df.head(500), max_steps=20000)

# Coleta latências
latencies_main = np.array([
    req.finish_step - req.arrival_step
    for req in runner_main.scheduler.completed
])

# Percentis detalhados
percentiles = evaluator.calculate_latency_percentiles(latencies_main)
print('Percentis de Latência (config: batch=16, max_tokens=4096):')
for k, v in percentiles.items():
    print(f'  {k}: {v:.1f} steps')

In [ ]:
# Visualização com SLO
fig = evaluator.plot_latency_distribution(
    latencies_main,
    title='Serving com Continuous Batching (batch=16)',
    slo_p99=500.0,
    save_name='03_latency_distribution.png',
)
plt.show()

## 4. Análise de Memória: ZeRO e Capacity Planning

Conforme seção *ZeRO: Otimização de Data Parallelism* (Rajbhandari et al., 2020):

> *"Para um modelo com $\Psi$ parâmetros e $N_d$ dispositivos, com ZeRO Stage 3
> cada dispositivo armazena apenas $\frac{K \cdot \Psi}{N_d}$ bytes."*

Vamos calcular requisitos de memória para diferentes modelos e estratégias.

In [ ]:
# Capacity planning: memória por modelo e estratégia
models = {
    'LLaMA-2 7B': {'params_b': 7, 'layers': 32, 'hidden': 4096, 'heads': 32},
    'LLaMA-2 13B': {'params_b': 13, 'layers': 40, 'hidden': 5120, 'heads': 40},
    'LLaMA-2 70B': {'params_b': 70, 'layers': 80, 'hidden': 8192, 'heads': 64},
}

K_adam_fp16 = 16  # bytes por parâmetro (Adam + FP16 mixed precision)
K_inference_fp16 = 2  # bytes por parâmetro (apenas pesos FP16)

print('Requisitos de Memória (conforme ZeRO — Rajbhandari et al., 2020):')
print('=' * 80)

capacity_data = []
for name, cfg in models.items():
    psi = cfg['params_b'] * 1e9
    
    for strategy, n_devices in [('Single GPU', 1), ('ZeRO-3 (2 GPUs)', 2),
                                 ('ZeRO-3 (4 GPUs)', 4), ('ZeRO-3 (8 GPUs)', 8)]:
        # Memória de parâmetros
        if 'ZeRO' in strategy:
            mem_params_gb = (K_inference_fp16 * psi / n_devices) / 1e9
        else:
            mem_params_gb = (K_inference_fp16 * psi) / 1e9
        
        capacity_data.append({
            'model': name,
            'strategy': strategy,
            'n_devices': n_devices,
            'mem_per_device_gb': round(mem_params_gb, 1),
            'fits_24gb': '✓' if mem_params_gb <= 24 else '✗',
            'fits_80gb': '✓' if mem_params_gb <= 80 else '✗',
        })

cap_df = pd.DataFrame(capacity_data)
cap_df

In [ ]:
# Visualização de capacity planning
fig, ax = plt.subplots(figsize=(12, 6))

for model_name in models:
    subset = cap_df[cap_df['model'] == model_name]
    ax.plot(subset['n_devices'], subset['mem_per_device_gb'], 'o-',
            linewidth=2, markersize=8, label=model_name)

ax.axhline(24, color='orange', linestyle='--', alpha=0.7, label='A10G (24GB)')
ax.axhline(80, color='red', linestyle='--', alpha=0.7, label='A100 (80GB)')
ax.set_xlabel('Número de Dispositivos (ZeRO Stage 3)', fontsize=12)
ax.set_ylabel('Memória por Dispositivo (GB)', fontsize=12)
ax.set_title('Capacity Planning: Memória de Parâmetros\n'
             'ZeRO Stage 3 (Rajbhandari et al., 2020)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
ax.set_xticks([1, 2, 4, 8])

plt.tight_layout()
plt.savefig('../outputs/figures/03_capacity_planning.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tabela Comparativa de Estratégias de Paralelismo

Reproduz e expande a **Tabela 1** do material da aula:

> *"Fonte: Elaborado pelo autor, baseado em Narayanan et al. (2021)"*

In [ ]:
# Tabela comparativa expandida (conforme Tabela 1 do material)
strategies = pd.DataFrame({
    'Estratégia': ['Data Parallelism', 'Tensor Parallelism',
                   'Pipeline Parallelism', 'ZeRO-Inference',
                   'Hybrid (TP+PP)', '3D Parallelism'],
    'Caso de Uso': [
        'Modelo cabe em 1 GPU',
        'Modelo grande, baixa latência',
        'Modelos muito grandes',
        'Memória limitada',
        'Modelos 70B+ intra+inter nó',
        'Modelos 100B+ clusters grandes',
    ],
    'Overhead Comunicação': ['Baixo', 'Médio', 'Baixo-Médio',
                             'Médio', 'Médio-Alto', 'Alto'],
    'Complexidade': ['Baixa', 'Média', 'Alta', 'Média', 'Alta', 'Muito Alta'],
    'Escalabilidade': ['Linear', '8-16 GPUs', 'Alta', 'Alta',
                        'Dezenas de GPUs', 'Centenas+'],
    'Framework': ['PyTorch DDP', 'Megatron-LM', 'GPipe/PipeDream',
                  'DeepSpeed', 'Megatron+DeepSpeed', 'DeepSpeed 3D'],
})

print('Tabela Comparativa de Estratégias de Paralelismo (Narayanan et al., 2021):')
strategies

## 6. Análise de Sensibilidade: Token Budget

Conforme a discussão sobre PagedAttention e continuous batching,
o "token budget" (limite de tokens totais em memória) é um parâmetro
crítico que determina quantas requisições podem ser processadas
simultaneamente.

In [ ]:
# Sensibilidade ao token budget
token_budgets = [1024, 2048, 4096, 8192, 16384]
budget_results = []

for budget in token_budgets:
    config = ServingConfig(max_batch_size=32, max_total_tokens=budget)
    runner = InferenceRunner(config)
    result = runner.run_continuous_batching(df.head(300), max_steps=20000)
    
    latencies = np.array([
        req.finish_step - req.arrival_step
        for req in runner.scheduler.completed
    ])
    
    budget_results.append({
        'token_budget': budget,
        'total_steps': result['total_steps'],
        'avg_latency': result['avg_latency_steps'],
        'p99_latency': result['p99_latency_steps'],
        'throughput': result['throughput_req_per_step'],
    })

budget_df = pd.DataFrame(budget_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(budget_df['token_budget'], budget_df['p99_latency'], 'o-',
        color='red', linewidth=2, markersize=8, label='P99')
ax.plot(budget_df['token_budget'], budget_df['avg_latency'], 's-',
        color='steelblue', linewidth=2, markersize=8, label='Média')
ax.set_xlabel('Token Budget')
ax.set_ylabel('Latência (steps)')
ax.set_title('Latência vs. Token Budget')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(budget_df['token_budget'], budget_df['throughput'], 'o-',
        color='seagreen', linewidth=2, markersize=8)
ax.set_xlabel('Token Budget')
ax.set_ylabel('Throughput (req/step)')
ax.set_title('Throughput vs. Token Budget')
ax.grid(True, alpha=0.3)

plt.suptitle('Sensibilidade ao Token Budget\n'
             '(gestão de memória KV-Cache — Kwon et al., 2023)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/03_token_budget_sensitivity.png', dpi=150,
            bbox_inches='tight')
plt.show()

budget_df.round(3)

## 7. Resumo Executivo e Recomendações

Síntese dos resultados obtidos, conectados com as conclusões do material da aula.

In [ ]:
# Resumo executivo
print('=' * 60)
print('RESUMO EXECUTIVO — Aula 07')
print('Inferência Distribuída e Paralelismo de Modelos')
print('=' * 60)

print('\n1. PARALELISMO:')
print('   - Tensor Parallelism: eficiente para 2-8 GPUs intra-nó (NVLink)')
print('   - Pipeline Parallelism: bubble fraction → 0 com M >> P micro-batches')
print('   - Hybrid TP+PP: necessário para modelos >70B em clusters')

print('\n2. SCHEDULING:')
print('   - Continuous batching superior ao static em todos os cenários')
print('   - Maior benefício com alta variabilidade de comprimento de saída')

print('\n3. MEMÓRIA:')
print('   - PagedAttention: >95% utilização vs. ~50% contígua')
print('   - Prefix sharing: economia proporcional ao nº de conversas')
print('   - ZeRO-3: permite modelos de qualquer tamanho com GPUs suficientes')

print('\n4. LATÊNCIA:')
print('   - Speculative decoding: 2-3x speedup com α ≈ 80%')
print('   - Trade-off batch_size: maior batch → maior throughput mas maior P99')

print('\n5. SLOs (conforme material da aula):')
print('   - P99 latência < 500ms para aplicações interativas')
print('   - Auto-scaling baseado em queue depth + utilização de GPU')
print('   - Pré-aquecimento de instâncias standby para picos')

In [ ]:
# Salva métricas finais
final_metrics = {
    'slo_analysis': slo_df.to_dict('records'),
    'capacity_planning': cap_df.to_dict('records'),
    'token_budget_sensitivity': budget_df.to_dict('records'),
}

save_metrics(
    final_metrics,
    '../outputs/logs/evaluation_results.json',
    metadata={'notebook': '03_avaliacao', 'workload_size': len(df)},
)
print('Métricas de avaliação salvas em outputs/logs/evaluation_results.json')

## Resumo

Neste notebook avaliamos:

1. **SLOs e percentis** — P50/P95/P99 para diferentes configurações de batch
2. **Trade-off latência vs. throughput** — batch_size controla o equilíbrio
3. **Capacity planning com ZeRO** — memória ∝ params/N_devices (Rajbhandari et al., 2020)
4. **Token budget** — parâmetro crítico para continuous batching + PagedAttention
5. **Estratégias comparativas** — Tabela expandida de Narayanan et al. (2021)

### Conexão com Hands-On do Material

Os conceitos explorados nestes notebooks conectam-se diretamente com a prática
descrita no material da aula: configuração de vLLM com tensor parallelism,
continuous batching com PagedAttention, e benchmark comparativo mostrando
throughput de 40 req/s (vLLM) vs. 5 req/s (HF vanilla) — melhoria de 8×.